In [ ]:
import os, sys
from dotenv import load_dotenv
from agents import Agent, Runner, FileSearchTool, function_tool, trace
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

# The notebook runs from notebooks/, so add the project root to the path
sys.path.insert(0, os.path.abspath(".."))

%load_ext autoreload
%autoreload 2

In [ ]:
load_dotenv(override=True)
MODEL = os.getenv("MODEL", "gpt-4.1-mini")

In [ ]:
# Check the key - if you're not using OpenAI, check whichever key you're using! Ollama doesn't need a key.

import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")
    


In [ ]:
VECTOR_STORE_ID = os.getenv("VECTOR_STORE_ID")

In [ ]:
# Use a SEPARATE notebook database (notebooks/notebooks_test.sqlite3), never the real chatbot.sqlite3.
# It must run before anything from chatbot/ is imported (restart the kernel if you already imported it).
from notebooks.notebook_db import use_test_database, all_leads, all_demo_requests, clear_leads_and_demos
use_test_database()

from chatbot.prompts import STABLE_FACTS, KNOWLEDGE_RULES, NEVER_RULES, TONE
from chatbot.tools import check_course_intake, save_lead, request_opendoor_demo
print("Loaded:", MODEL, VECTOR_STORE_ID)

In [ ]:
print(STABLE_FACTS[:80])
print(check_course_intake.name, save_lead.name, request_opendoor_demo.name)

# Test set

In [ ]:
# Cell 2: Test set — the same questions are used for both patterns, so the comparison is fair
test_questions = [
    # Services & Sales
    ("services", "Can you build a school website for us?"),
    ("services", "Do you do AI chatbots for businesses?"),

    # Academy
    ("academy",  "When does the web dev bootcamp start?"),
    ("academy",  "I'm a beginner, which course suits me?"),

    # OpenDoor
    ("opendoor", "My church wants to try OpenDoor"),
    ("opendoor", "I can't log in to OpenDoor"),

    # Mixed / general
    ("mixed",    "I want to learn web development and also need a site for my church"),
    ("general",  "Where are you located?"),
]

for area, q in test_questions:
    print(f"[{area:8}] {q}")
print(f"\n{len(test_questions)} test questions ready")

# Filtered file search.

In [ ]:
# Cell 3: one FileSearchTool per specialist, same vector store, filtered by domain

def domain_filter(*domains):
    """Only search chunks whose 'domain' attribute is one of these values."""
    return {
        "type": "or",
        "filters": [{"type": "eq", "key": "domain", "value": d} for d in domains],
    }

services_search = FileSearchTool(
    vector_store_ids=[VECTOR_STORE_ID],
    max_num_results=5,
    include_search_results=True,
    filters=domain_filter("services", "company"),
)

academy_search = FileSearchTool(
    vector_store_ids=[VECTOR_STORE_ID],
    max_num_results=5,
    include_search_results=True,
    filters=domain_filter("academy", "company"),
)

opendoor_search = FileSearchTool(
    vector_store_ids=[VECTOR_STORE_ID],
    max_num_results=5,
    include_search_results=True,
    filters=domain_filter("opendoor", "company"),
)

print("3 filtered search tools ready")

In [ ]:
# Cell 4: Specialist instructions = shared blocks + the specialist's own section

def build_instructions(role_section: str) -> str:
    """Put the shared Coltek blocks together with one specialist's role."""
    return "\n\n".join([
        RECOMMENDED_PROMPT_PREFIX,   # tells the agent how handoffs work (it will receive them)
        STABLE_FACTS,
        KNOWLEDGE_RULES,
        NEVER_RULES,
        TONE,
        role_section,
    ])


SERVICES_ROLE = """
# YOUR ROLE: SERVICES & SALES SPECIALIST

You handle visitors who want Coltek Technologies to build something or train their team: custom software, websites and web apps, mobile apps, AI solutions such as chatbots and document assistants, school or church systems, and corporate technology training.
Understand the visitor's need before explaining services. Ask about: what they want to build or improve, who will use it, must-have features, any deadline, and a budget range only if they are willing to share. Ask at most two questions per message.
Explain relevant services in plain language and mention relevant past work from the portfolio when it fits.
Pricing: Coltek quotes per project after a consultation. Never give numbers of your own.
When the visitor shows clear interest, offer to have the team contact them. Collect name and phone or WhatsApp, read the details back, ask "May Coltek contact you about this by phone, WhatsApp or email?", and call save_lead with interest "services" only after a clear yes. Quote the reference it returns."""

ACADEMY_ROLE = """
# YOUR ROLE: CODE ACADEMY ADVISOR

You handle learners interested in Coltek Code Academy. Confirmed programmes: Complete Web Development Bootcamp and UX/UI Design Masterclass. Coltek also trains in AI, Python, cybersecurity, Microsoft Office, Google Workspace and computer literacy.
Help learners choose by asking about their current level, their goal (job, freelance, business, school) and their available time.
For dates, fees, duration, mode, seats or whether a course is open, always call check_course_intake. A null or missing value means "not confirmed yet". Say so and offer to register their interest.
Mention that web development applicants may take a friendly 45–60 minute placement assessment that is not pass-or-fail.
Never promise jobs or accredited certification. Never share the student WhatsApp group link.
To register interest or enrolment intent, collect name and phone, read them back, get a clear yes to being contacted, then call save_lead with interest "academy".
If the visitor has already chosen a course and says they want to join, skip the course-choice questions and go straight to collecting name and phone.
Before calling save_lead, always read the name and phone back in one line and get the visitor's yes, even if they already gave consent.
If save_lead returns DUPLICATE_LEAD, tell the visitor their interest is already registered and quote the existing reference."""

OPENDOOR_ROLE = """
# YOUR ROLE: OPENDOOR SPECIALIST

You handle churches evaluating or using OpenDoor, Coltek's cloud church management system at open-door.site.
For churches evaluating it: explain features only from the knowledge files. Ask the church name, approximate membership size, location and main challenge. If pricing is not in the files, say pricing is shared during a demo. Offer a demo and call request_opendoor_demo after collecting details and getting a clear yes to being contacted.
For existing users: members who cannot log in should contact their own church administrator first, because accounts are created per church. Administrators can email support@open-door.site.
Never ask for or accept passwords. Never confirm whether any person or email exists in a church's records.
For suspected data loss, another church's data being visible, giving or payment errors, or a full outage, give support@open-door.site and the phone number immediately and tell them it is urgent.
A normal login problem is not urgent; do not give the phone number for it."""


services_instructions = build_instructions(SERVICES_ROLE)
academy_instructions  = build_instructions(ACADEMY_ROLE)
opendoor_instructions = build_instructions(OPENDOOR_ROLE)

# Quick check: each prompt should have all six parts
for name, text in [("services", services_instructions),
                   ("academy", academy_instructions),
                   ("opendoor", opendoor_instructions)]:
    print(f"{name:9} {len(text):5} chars | "
          f"facts={'STABLE FACTS' in text} never={'NEVER' in text} role={'YOUR ROLE' in text}")

# 3 Agen specialists, each with its own filtered search and tools

In [ ]:
services_agent = Agent(
    name="Services & Sales Specialist",
    handoff_description="Handles requests for Coltek to build software, websites, mobile apps, AI solutions, school/church systems, or run corporate training, plus quotes.",
    instructions=services_instructions,
    model=MODEL,
    tools=[services_search, save_lead],
)

academy_agent = Agent(
    name="Code Academy Advisor",
    handoff_description="Handles Coltek Code Academy courses: choosing a course, enrolment, dates, fees, beginners, assessments and certificates.",
    instructions=academy_instructions,
    model=MODEL,
    tools=[academy_search, check_course_intake, save_lead],
)

opendoor_agent = Agent(
    name="OpenDoor Specialist",
    handoff_description="Handles OpenDoor church management software: features, demos, pricing, login problems and support.",
    instructions=opendoor_instructions,
    model=MODEL,
    tools=[opendoor_search, request_opendoor_demo],
)

for a in (services_agent, academy_agent, opendoor_agent):
    print(f"{a.name:28} tools={[t.name for t in a.tools]}")

# Test Agent Before Orchestration

In [ ]:
# Test each specialist directly before orchestrating
solo_tests = [
    (services_agent, "Can you build a school website for us?"),
    (academy_agent,  "When does the web dev bootcamp start?"),
    (opendoor_agent, "I can't log in to OpenDoor"),
]

with trace("Specialists - solo tests"):
    for agent, q in solo_tests:
        r = await Runner.run(agent, q)
        print(f"=== {agent.name} ===\nQ: {q}\nA: {r.final_output}\n")

Now go and check out the trace:

https://platform.openai.com/traces

Each specialist becomes a tool the manager can call

In [ ]:


services_tool = services_agent.as_tool(
    tool_name="ask_services_specialist",
    tool_description="Send questions about Coltek building something: websites, web or mobile apps, custom software, AI solutions, school/church systems, corporate training, or quotes. Include the visitor's full question and any details they gave.",
)

academy_tool = academy_agent.as_tool(
    tool_name="ask_academy_specialist",
    tool_description="Send questions about Coltek Code Academy: which course suits someone, enrolment, start dates, fees, duration, beginners, assessments or certificates. Include the visitor's full question and any details they gave.",
)

opendoor_tool = opendoor_agent.as_tool(
    tool_name="ask_opendoor_specialist",
    tool_description="Send questions about OpenDoor church software: features, demos, pricing, login problems, support or data issues. Include the visitor's full question and any details they gave.",
)

print([t.name for t in (services_tool, academy_tool, opendoor_tool)])

In [ ]:
opendoor_tool

# The manager

The manager has no file search of its own, so it must delegate to the specialists.

In [ ]:
MANAGER_ROLE = """
# HOW YOU WORK

You do not answer service, course or OpenDoor questions yourself. You consult specialists and then write one clear reply to the visitor.
ask_services_specialist: projects, websites, apps, AI solutions, school/church systems, corporate training, quotes.
ask_academy_specialist: courses, enrolment, dates, fees, beginners, assessments, certificates.
ask_opendoor_specialist: OpenDoor features, demos, pricing, login and support.
When you call a specialist, pass the visitor's full question plus any details they have already given (name, organisation, what they need), because the specialist cannot see the conversation.
If a question covers two areas, consult both specialists and combine their answers into one reply.
General company questions (location, contacts, who we are) you answer from STABLE FACTS without calling anyone.
Never reveal that specialists exist. Speak as one assistant."""

manager_instructions = "\n\n".join([
    'You are "Coltek Assistant", the website assistant for Coltek Technologies in Accra, Ghana.',
    STABLE_FACTS,    # fills {STABLE FACTS}
    NEVER_RULES,     # fills {NEVER}
    TONE,            # fills {TONE}
    MANAGER_ROLE,
])

manager_agent = Agent(
    name="Coltek Assistant (Manager)",
    instructions=manager_instructions,
    model=MODEL,
    tools=[services_tool, academy_tool, opendoor_tool], 
)

print("Manager tools:", [t.name for t in manager_agent.tools])

# The three-turn lead conversation

In [ ]:
# Weakness test: does the manager pass enough context to the Academy specialist?
import json
from agents import ToolCallItem

def show_specialist_calls(result):
    """Print exactly what the manager sent to each specialist this turn."""
    for item in result.new_items:
        if isinstance(item, ToolCallItem) and item.raw_item.name.startswith("ask_"):
            args = json.loads(item.raw_item.arguments)
            print(f"   📨 {item.raw_item.name} received: {args.get('input', args)}")

turns = [
    "I want to join the web dev bootcamp.",
    "My name is Kwame Mensah.",
    "My phone number is 0541234567. Yes, you can contact me.",
]

leads_before = len(all_leads())
history = []

with trace("4a Manager - weakness: 3-turn lead"):
    for i, msg in enumerate(turns, 1):
        history = history + [{"role": "user", "content": msg}]
        r = await Runner.run(manager_agent, history)
        print(f"--- Turn {i} ---")
        print(f"Visitor: {msg}")
        show_specialist_calls(r)
        print(f"Assistant: {r.final_output}\n")
        history = r.to_input_list()   # the manager sees the whole conversation next turn

print("New leads saved:", all_leads()[leads_before:])

# tighten the instruction and rebuild the manager

In [ ]:
# Tighten the "pass full details" rule
MANAGER_ROLE = MANAGER_ROLE.replace(
    "When you call a specialist, pass the visitor's full question plus any details they have already given (name, organisation, what they need), because the specialist cannot see the conversation.",
    """When you call a specialist, the specialist CANNOT see this conversation. Every call must be self-contained. Always include:
- the visitor's current message, word for word;
- the topic so far (e.g. "wants to join the Complete Web Development Bootcamp");
- every detail the visitor has given in ANY earlier turn: name, phone/WhatsApp, email, organisation, preferred contact method;
- whether the visitor has clearly said YES to being contacted (quote their words), or that consent has not been given yet;
- what you already told them (e.g. "the read-back of their details was shown and they confirmed").
Never send just the latest message on its own."""
)

manager_instructions = "\n\n".join([
    'You are "Coltek Assistant", the website assistant for Coltek Technologies in Accra, Ghana.',
    STABLE_FACTS, NEVER_RULES, TONE, MANAGER_ROLE,
])
manager_agent = Agent(
    name="Coltek Assistant (Manager)",
    instructions=manager_instructions,
    model=MODEL,
    tools=[services_tool, academy_tool, opendoor_tool],
)
print("Rule updated:", "CANNOT see this conversation" in manager_instructions)

In [ ]:
# Weakness test: does the manager pass enough context to the Academy specialist?
import json
from agents import ToolCallItem

def show_specialist_calls(result):
    """Print exactly what the manager sent to each specialist this turn."""
    for item in result.new_items:
        if isinstance(item, ToolCallItem) and item.raw_item.name.startswith("ask_"):
            args = json.loads(item.raw_item.arguments)
            print(f"   📨 {item.raw_item.name} received: {args.get('input', args)}")

turns = [
    "I want to join the web dev bootcamp.",
    "My name is Kwame Mensah.",
    "My phone number is 0541234567. Yes, you can contact me.",
]

leads_before = len(leads)
history = []

with trace("4a Manager - weakness: 3-turn lead"):
    for i, msg in enumerate(turns, 1):
        history = history + [{"role": "user", "content": msg}]
        r = await Runner.run(manager_agent, history)
        print(f"--- Turn {i} ---")
        print(f"Visitor: {msg}")
        show_specialist_calls(r)
        print(f"Assistant: {r.final_output}\n")
        history = r.to_input_list()   # the manager sees the whole conversation next turn

print("New leads saved:", leads[leads_before:])

# via Handoffs

With handoffs, the triage agent steps aside and the specialist takes over the conversation and talks to the visitor directly.

In [ ]:
HANDBACK_LINE = (
    "Only answer the part of the visitor's request that is in your area. "
    "If they also need something outside your area (now or later in the conversation), "
    "finish your part, then hand back to the Coltek Assistant so the right specialist can help with the rest."
)

# 1. Copies of the specialists for 4b. build_instructions() already starts with RECOMMENDED_PROMPT_PREFIX

services_4b = services_agent.clone(
    name="Services and Sales Specialist",   # no '&' -> valid handoff tool name, no warning
    instructions=build_instructions(SERVICES_ROLE + "\n" + HANDBACK_LINE),
)
academy_4b  = academy_agent.clone(instructions=build_instructions(ACADEMY_ROLE  + "\n" + HANDBACK_LINE))
opendoor_4b = opendoor_agent.clone(instructions=build_instructions(OPENDOOR_ROLE + "\n" + HANDBACK_LINE))

# 2. Triage instructions
TRIAGE_ROUTING = """# ROUTING
Work out what the visitor needs within your first one or two replies, then hand off:
Services & Sales specialist: building software, websites, apps, AI solutions, school/church systems, corporate training, quotes.
Code Academy advisor: learning, courses, enrolment, dates, fees, beginners.
OpenDoor specialist: church management software, demos, OpenDoor login or support.
Any question about what Coltek builds, sells, teaches or supports — even a simple "do you do X?" — belongs to a specialist. Hand off immediately; do not answer it yourself and do not ask permission to hand off.
If unclear, ask one short question such as "Are you looking to build something, learn a skill, or use OpenDoor for your church?"
Answer general company questions (location, contacts, who we are) yourself without handing off.
Hand off quietly. Do not say "I am transferring you"."""

triage_instructions = "\n\n".join([
    RECOMMENDED_PROMPT_PREFIX,     # must come first: teaches the model how handoffs work
    'You are "Coltek Assistant", the front door of the Coltek Technologies website.',
    STABLE_FACTS,
    NEVER_RULES,
    TONE,
    TRIAGE_ROUTING,
])

# 3. Triage hands off to the three specialists
triage_agent = Agent(
    name="Coltek Assistant",
    instructions=triage_instructions,
    model=MODEL,
    handoffs=[services_4b, academy_4b, opendoor_4b],
)

# 4. Each specialist can hand back to triage (set after, because triage must exist first)
for spec in (services_4b, academy_4b, opendoor_4b):
    spec.handoffs = [triage_agent]

# Check
print("Triage hands off to:", [a.name for a in triage_agent.handoffs])
for spec in (services_4b, academy_4b, opendoor_4b):
    print(f"{spec.name:28} -> back to {[a.name for a in spec.handoffs]} | "
          f"prefix={spec.instructions.startswith(RECOMMENDED_PROMPT_PREFIX)} "
          f"handback={HANDBACK_LINE in spec.instructions}")

In [ ]:
with trace("4b Handoffs - test set"):
    for area, q in test_questions:
        r = await Runner.run(triage_agent, q)
        print(f"[{area:8}] {q}\n   answered by: {r.last_agent.name}\n   -> {r.final_output}\n")

In [ ]:
# 4b test: does each question land on the right agent?

# Which agent should answer each area of the test set
expected_agent = {
    "services": "Services and Sales Specialist",
    "academy":  "Code Academy Advisor",
    "opendoor": "OpenDoor Specialist",
    "general":  "Coltek Assistant",      # triage keeps general company questions
    "mixed":    None,                    # either specialist is acceptable
}

passed = 0
with trace("4b handoffs"):
    for area, q in test_questions:
        result = await Runner.run(triage_agent, q)
        who = result.last_agent.name
        want = expected_agent[area]
        ok = (want is None) or (who == want)
        passed += ok

        print(f"{'✅' if ok else '❌'} [{area:8}] {q}")
        print(f"   answered by: {who}" + ("" if ok else f"   (expected: {want})"))
        print(f"   -> {result.final_output}\n")

print(f"Routing score: {passed}/{len(test_questions)}")

In [ ]:
# Cell 12: Multi-turn with handoffs. The visitor stays with whoever is handling them

turns = [
    "I want to join the web dev bootcamp.",
    "My name is Kwame Mensah.",
    "My phone number is 0541234567. Yes, you can contact me.",
    "Yes, that's correct.",     # confirms the read-back
]

# Start clean so earlier tests (e.g. the 4a run) don't cause DUPLICATE_LEAD.
# Start clean: delete the leads and demo requests from the NOTEBOOK test database
clear_leads_and_demos()
leads_before = len(all_leads())
current_agent = triage_agent   # the first turn always starts at the front door
history = []

with trace("Multi-turn handoffs - 3-turn lead"):
    for i, msg in enumerate(turns, 1):
        history = history + [{"role": "user", "content": msg}]
        result = await Runner.run(current_agent, history)

        print(f"--- Turn {i} ---")
        print(f"Visitor: {msg}")
        print(f"Started with: {current_agent.name}  ->  answered by: {result.last_agent.name}")
        print(f"Assistant: {result.final_output}\n")

        # The key lines: carry on with the agent that replied, and with the full history
        current_agent = result.last_agent
        history = result.to_input_list()

print("New leads saved:", all_leads()[leads_before:])

In [ ]:
for lead in all_leads():
    print(lead.reference, lead.full_name, lead.phone, lead.interest)

# The 4b weakness: one question, two areas

In [ ]:
# Cell 13: does a handoff handle a question that covers two areas?
from agents import HandoffOutputItem

def show_handoffs(result):
    """Print every handoff that happened in this run."""
    hops = [i for i in result.new_items if isinstance(i, HandoffOutputItem)]
    if not hops:
        print("   🔀 no handoff")
    for h in hops:
        print(f"   🔀 {h.source_agent.name}  ->  {h.target_agent.name}")

mixed_q = "I want to learn web development and also need a site for my church"

with trace("4b handoffs - weakness: mixed question"):
    # Turn 1: the mixed question
    r1 = await Runner.run(triage_agent, mixed_q)
    print("--- Turn 1 ---")
    print("Visitor:", mixed_q)
    show_handoffs(r1)
    print("Answered by:", r1.last_agent.name)
    print("Assistant:", r1.final_output, "\n")

    # Turn 2: the visitor pushes on the second half
    follow_up = "Thanks. And what about the church website?"
    r2 = await Runner.run(r1.last_agent, r1.to_input_list() + [{"role": "user", "content": follow_up}])
    print("--- Turn 2 ---")
    print("Visitor:", follow_up)
    show_handoffs(r2)
    print("Answered by:", r2.last_agent.name)
    print("Assistant:", r2.final_output)

# Handoff callback: logging which department got the conversation

In [ ]:
# Cell 14: log every handoff with the department and time
from datetime import datetime, timezone
from agents import handoff, RunContextWrapper

handoff_log = []   # in Part 6 this becomes a database table or a Google Sheet

def make_logged_handoff(agent, department):
    """Wrap a specialist so every handoff to it is logged."""
    def on_handoff(ctx: RunContextWrapper[None]) -> None:
        now = datetime.now(timezone.utc)          # Ghana uses GMT/UTC all year
        handoff_log.append({"department": department, "time": now.isoformat(timespec="seconds")})
        print(f"📋 handed to {department} at {now:%H:%M}")
    return handoff(agent=agent, on_handoff=on_handoff)

# Replace triage's plain handoffs with logged ones (same specialists, same routing)
triage_agent.handoffs = [
    make_logged_handoff(services_4b, "Services"),
    make_logged_handoff(academy_4b,  "Academy"),
    make_logged_handoff(opendoor_4b, "OpenDoor"),
]

# Test
with trace("4b handoffs - callback test"):
    for q in ["When does the web dev bootcamp start?",
              "My church wants to try OpenDoor",
              "Where are you located?"]:
        r = await Runner.run(triage_agent, q)
        print(f"   {q}  ->  {r.last_agent.name}\n")

print("Handoff log:", handoff_log)

# Production design decision

## What the tests showed

| | 4a Manager (agents as tools) | 4b Handoffs (triage → specialist) |
|---|---|---|
| Single-topic questions | Correct, but every answer goes through the manager | 8/8 routed to the right specialist after fixes |
| Multi-turn lead capture | ❌ Weak: specialist starts fresh each call; depends on the manager passing name, phone and consent | ✅ Strong: the Academy advisor saw the full history and saved the lead |
| Mixed question (course + church website) | ✅ Calls both specialists, one combined reply | ⚠️ Only one specialist at a time; other half needs a hand-back and extra turns |
| Cost / speed | Higher: manager + specialist both run every turn | Lower: after the handoff only the specialist runs |
| Voice | Always one voice (the manager writes every reply) | Specialist talks directly; tone kept consistent by shared TONE block |

## Decision

**I will build the hybrid in `chatbot/agents/`:**
- **Main structure: handoffs.** A triage agent ("Coltek Assistant") hands off to the Services, Academy and OpenDoor specialists. Most visitors have one need, and lead capture runs over several turns, which handoffs handle best.
- **Plus the specialists as tools on triage.** Triage can answer a quick question that touches another area (e.g. "do you also teach web dev?") without handing the whole conversation away.
- Specialists hand back to triage when the visitor's need moves outside their area.
- Every handoff is logged with department and time (the `on_handoff` callback from Cell 14) → used in Part 6 for reporting.

## Files in `chatbot/agents/`
- `specialists.py`: services, academy, opendoor agents (instructions from `chatbot/prompts.py`, tools from `chatbot/tools.py`, filtered file search)
- `triage.py`: triage agent with handoffs + specialist tools; exports `coltek_agent`
- `server/chat_service.py` (Part 6) imports the agents and runs them, always continuing with `result.last_agent`

## Still to watch
- Mixed questions: confirm the specialist hands back instead of answering outside its area.
- Leads live in a Python list for now. They need a real database or Sheet in Part 6.